# 08 — SR-GNN trên đồ thị nhóm timestamp (mô hình neural đối chứng)

Theo tài liệu tổng quan phương pháp, SR-GNN là GNN duy nhất ngang VSTAN trong benchmark độc lập (ECIR 2024), nên được dùng làm mô hình neural đối chứng cho GRU4Rec.

- Đồ thị phiên của một query (protocol v1.2.0): nút = item phân biệt trong 50 nhóm gần nhất của prefix (item ngoài vocabulary dùng chung nút UNK); input của nút = embedding item + trung bình embedding các hành vi của nút.
- Cạnh có hướng từ **mọi item của nhóm g sang mọi item của nhóm g+1**; không tạo thứ tự giả giữa các item cùng nhóm. Chuẩn hóa theo bậc vào/ra như SR-GNN gốc.
- Một bước gated GNN; readout: quan tâm cục bộ = trung bình trạng thái các nút của nhóm cuối, quan tâm toàn cục = attention trên mọi nút.
- Loss: sampled softmax như GRU4Rec (2048 negative dùng chung, hiệu chỉnh log-Q, nhãn `1/|G|`). Early stopping theo NDCG@20 validation.
- Đầu ra: `models/srgnn/{config.json, model.pt}`, `reports/eval/srgnn_validation.json`. **Không đọc test.**

## 1. Đường dẫn, giao thức và kiểm tra đầu vào

In [ ]:
from pathlib import Path
import sys, time

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src/rr").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from rr.io import environment, fingerprint, read_json, rel, require, utc_now, verify_inputs, write_json
from rr.candidates import CandidateFilter
from rr.samples import prefix_events, build_vocab
from rr.models import popularity, prefix_matrix
from rr.evaluation import load_samples, run, flat
from rr.gru import idx_to_itemid, topn_items
from rr.srgnn import GraphData, SRGNN, fit, negative_distribution, session_vectors, target_csr

PROTOCOL_PATH = ROOT / "configs/protocol.json"
PROTOCOL = read_json(PROTOCOL_PATH)
PROCESSED = ROOT / "data/processed"
EVAL_DIR = ROOT / "reports/eval"
MODEL_DIR = ROOT / "models/srgnn"
FALLBACK_TOP_N = PROTOCOL["candidates"]["fallback_top_n"]
MAX_GROUPS = PROTOCOL["knn"]["max_prefix_groups"]
SEED = PROTOCOL["seed"]
WIDTH = 200
torch.set_num_threads(4)

samples_report = read_json(ROOT / "reports/samples/metrics.json")
metadata_report = read_json(ROOT / "reports/metadata/metrics.json")
split_report = read_json(ROOT / "reports/session_split/metrics.json")
pop_report = read_json(EVAL_DIR / "popularity_validation.json")
for name, r in [("samples", samples_report), ("popularity_validation", pop_report)]:
    require(r["protocol"]["version"] == PROTOCOL["version"], f"{name} được tạo với protocol khác.")
expected = {p: samples_report["artifacts"][p] for p in [
    "data/processed/item_vocab.parquet", "data/processed/samples/train_queries.parquet",
    "data/processed/samples/train_targets.parquet", "data/processed/samples/validation_queries.parquet",
    "data/processed/samples/validation_targets.parquet"]}
for p in ["data/processed/item_property_history.parquet", "data/processed/item_catalog.parquet"]:
    expected[p] = metadata_report["artifacts"][p]
expected["data/processed/events_sessionized.parquet"] = split_report["artifacts"]["data/processed/events_sessionized.parquet"]
inputs = verify_inputs(ROOT, {k: {"bytes": v["bytes"], "sha256": v["sha256"]} for k, v in expected.items()})
print("Đầu vào khớp SHA-256:", *inputs, sep="\n  ")
print(environment(), "| torch", torch.__version__, "| protocol", PROTOCOL["version"])

## 2. Ví dụ biên

Prefix `g1 = {A}`, `g2 = {B, C}` (cùng timestamp), `g3 = {A}`: cạnh A→B, A→C, B→A, C→A; **không** có cạnh B↔C; nhóm cuối = {A}; C chỉ có hành vi addtocart.

In [ ]:
_p = pd.DataFrame({"query_id": 0, "itemid": [1, 2, 3, 1], "event": ["view", "view", "addtocart", "view"], "distance": [2, 1, 1, 0]})
_g = GraphData(_p, [0], build_vocab(pd.DataFrame({"itemid": [1, 2, 3]})))
assert set(zip(_g.e_src.tolist(), _g.e_dst.tolist())) == {(0, 1), (0, 2), (1, 0), (2, 0)}
assert _g.l_val.tolist() == [0] and _g.n_beh[2].tolist() == [0, 1, 0]
del _p, _g
print("Đã kiểm tra đồ thị nhóm: cạnh giữa nhóm liên tiếp, không có thứ tự trong nhóm.")

## 3. Dữ liệu đồ thị cho train và validation

In [ ]:
vocab = pd.read_parquet(PROCESSED / "item_vocab.parquet")
catalog = pd.read_parquet(PROCESSED / "item_catalog.parquet")
history = pd.read_parquet(PROCESSED / "item_property_history.parquet")
events = pd.read_parquet(PROCESSED / "events_sessionized.parquet",
                         columns=["session_id", "group_index", "itemid", "event", "timestamp", "split"])
events["split"] = events["split"].astype(str)
train_events = events[events["split"] == "train"]
val_events = events[events["split"] == "validation"]
flt = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms")
flt_avail = CandidateFilter.from_frames(vocab, catalog, history, "known_at_train_ms", use_availability=True)
fb_items, fb_scores = (a[:FALLBACK_TOP_N] for a in popularity(train_events, pop_report["selected"]["fallback_weights"]))
train_q, train_t = load_samples(ROOT, "train")
val_q, val_t = load_samples(ROOT, "validation")

t0 = time.perf_counter()
train_prefix = prefix_events(train_q, train_events, ("itemid", "event"), max_groups=MAX_GROUPS)
train_graphs = GraphData(train_prefix, train_q["query_id"].to_numpy(), vocab)
pos_ptr, pos_val = target_csr(train_t, train_q["query_id"].to_numpy(), vocab)
del train_prefix
val_prefix = prefix_events(val_q, val_events, ("itemid", "event"), max_groups=MAX_GROUPS)
val_graphs = GraphData(val_prefix, val_q["query_id"].to_numpy(), vocab)
seen = prefix_matrix(val_prefix, val_q["query_id"].to_numpy())
n_items = int(vocab["item_idx"].max()) + 1
idx_item = idx_to_itemid(vocab)
neg_probs, log_q = negative_distribution(pos_val, n_items, alpha=0.5)
require(len(train_graphs) == len(train_q) and (np.diff(pos_ptr) > 0).all(), "Mỗi query train phải có target trong vocabulary.")
print(f"Chuẩn bị đồ thị: {time.perf_counter() - t0:.0f}s | train {len(train_graphs):,} | validation {len(val_graphs):,} "
      f"| nút TB {np.diff(train_graphs.n_ptr).mean():.2f} | cạnh TB {np.diff(train_graphs.e_ptr).mean():.2f}")

def evaluate(model, flt_=flt, exclude=None):
    t0 = time.perf_counter()
    items, scores = topn_items(model, session_vectors(model, val_graphs), idx_item, WIDTH)
    q_s = time.perf_counter() - t0
    report, *_ = run(items, scores, val_q, val_t, flt_, fb_items, fb_scores, exclude=exclude)
    report["query_seconds_total"], report["query_ms_per_query"] = round(q_s, 1), round(1000 * q_s / len(val_q), 4)
    return report

## 4. Smoke test và lưới siêu tham số

Smoke test: 1% query train, loss phải giảm. Lưới: `dim = 100` (như SR-GNN gốc), `lr ∈ {0,001; 0,003}`, batch 256, tối đa 6 epoch, dừng sớm sau 2 epoch không tăng.

In [ ]:
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
small_rows = rng.choice(len(train_graphs), len(train_graphs) // 100, replace=False)
class _Sub:  # view of 1% of the graphs with the same API
    def __init__(self, g, rows):
        self.g, self.rows = g, rows
        self.n_ptr = np.r_[0, np.cumsum(np.diff(g.n_ptr)[rows])]
    def __len__(self):
        return len(self.rows)
    def batch(self, r):
        return self.g.batch(self.rows[r])
_sub_ptr = np.r_[0, np.cumsum(np.diff(pos_ptr)[small_rows])]
_sub_val = np.concatenate([pos_val[pos_ptr[r]:pos_ptr[r + 1]] for r in small_rows])
smoke = SRGNN(n_items, dim=32)
smoke_hist = fit(smoke, _Sub(train_graphs, small_rows), _sub_ptr, _sub_val, neg_probs, log_q, epochs=3, lr=0.005,
                 batch_size=256, n_neg=512, seed=SEED)
display(pd.DataFrame(smoke_hist))
require(smoke_hist[-1]["loss"] < smoke_hist[0]["loss"], "Smoke test: loss không giảm.")
del smoke

runs, models = {}, {}
for lr in [0.001, 0.003]:
    name = f"dim100_lr{lr}"
    torch.manual_seed(SEED)
    model = SRGNN(n_items, dim=100, steps=1, dropout=0.1)
    reports = {}
    def on_epoch(epoch, model=model, name=name, reports=reports):
        reports[epoch] = evaluate(model)
        print(f"  {name} epoch {epoch}: ndcg@20={reports[epoch]['all']['ndcg@20']:.4f}", flush=True)
        return reports[epoch]["all"]["ndcg@20"]
    t0 = time.perf_counter()
    hist = fit(model, train_graphs, pos_ptr, pos_val, neg_probs, log_q, epochs=6, lr=lr, batch_size=256, n_neg=2048,
               seed=SEED, on_epoch=on_epoch, patience=2)
    best_epoch = max(reports, key=lambda e: reports[e]["all"]["ndcg@20"])
    runs[name] = {"config": {"dim": 100, "steps": 1, "dropout": 0.1, "lr": lr, "batch_size": 256, "n_neg": 2048,
                             "sample_alpha": 0.5, "max_groups": MAX_GROUPS},
                  "history": hist, "best_epoch": best_epoch, "report": reports[best_epoch],
                  "fit_seconds": round(time.perf_counter() - t0, 1)}
    models[name] = model
best_name = max(runs, key=lambda n: runs[n]["report"]["all"]["ndcg@20"])
best_model = models[best_name]
display(pd.DataFrame({n: {"ndcg@20": r["report"]["all"]["ndcg@20"], "hit@20": r["report"]["all"]["hit@20"],
                          "best_epoch": r["best_epoch"], "fit_min": r["fit_seconds"] / 60} for n, r in runs.items()}).T)

## 5. Biến thể báo cáo và lưu

In [ ]:
variants = {"main": runs[best_name]["report"], "exclude_seen": evaluate(best_model, exclude=seen),
            "with_availability": evaluate(best_model, flt_=flt_avail)}
for part in ["all", "new_targets", "repeat_targets", "strong_behavior_targets"]:
    print(part)
    display(flat(variants, part)[["hit@20", "recall@20", "mrr@20", "ndcg@20"]].style.format("{:.4f}"))

MODEL_DIR.mkdir(parents=True, exist_ok=True)
config = {"name": best_name, **runs[best_name]["config"], "n_items": n_items, "epochs": runs[best_name]["best_epoch"],
          "seed": SEED, "vocabulary": "train", "width": WIDTH}
torch.save(best_model.state_dict(), MODEL_DIR / "model.pt")
write_json(MODEL_DIR / "config.json", config)
require(all(fingerprint(ROOT / p) == v for p, v in inputs.items()), "Đầu vào thay đổi trong lúc chạy.")
write_json(EVAL_DIR / "srgnn_validation.json", {
    "generated_at_utc": utc_now(), **environment(), "torch_version": torch.__version__, "notebook": "notebooks/08_srgnn.ipynb",
    "protocol": {"path": rel(PROTOCOL_PATH, ROOT), "version": PROTOCOL["version"], **fingerprint(PROTOCOL_PATH)},
    "inputs": inputs, "split": "validation", "fit_data": "train", "n_queries": len(val_q),
    "fallback": pop_report["selected"]["fallback"], "smoke_test": smoke_hist, "runs": runs, "selected": config,
    "model_file": {"path": rel(MODEL_DIR / "model.pt", ROOT), **fingerprint(MODEL_DIR / "model.pt")},
    "variants": variants, "status": "PASS"})
print("Báo cáo:", rel(EVAL_DIR / "srgnn_validation.json", ROOT))